In [1]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from pathlib import Path

In [2]:
RAW = Path("C:/Users/88019/Desktop/BFIPPDSR_project/data/raw/cic")
FILES = [
 "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
 "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
 "Friday-WorkingHours-Morning.pcap_ISCX.csv",
 "Monday-WorkingHours.pcap_ISCX.csv",
 "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
 "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
 "Tuesday-WorkingHours.pcap_ISCX.csv",
 "Wednesday-workingHours.pcap_ISCX.csv"
]

In [3]:
def load_one(fp):
    df = pd.read_csv(fp)
    df.columns = [c.strip() for c in df.columns]
    # Clean inf/NaN
    df = df.replace([np.inf,-np.inf], np.nan).dropna()
    # Drop non-feature columns if exist
    drop_cols = [c for c in ['Flow ID','Src IP','Src Port','Dst IP','Dst Port','Timestamp'] if c in df.columns]
    df = df.drop(columns=drop_cols, errors='ignore')
    # Standardize label
    label_col = 'Label' if 'Label' in df.columns else ('label' if 'label' in df.columns else None)
    if label_col is None:
        # Infer from filename
        name = fp.name.lower()
        is_attack = any(k in name for k in ['ddos','portscan','infilteration','webattacks'])
        df['Label'] = 1 if is_attack else 0
        label_col = 'Label'
    # Map to binary if strings
    if df[label_col].dtype == 'object':
        df[label_col] = df[label_col].str.lower().map(lambda x: 0 if 'benign' in x else 1)
    else:
        df[label_col] = df[label_col].astype(int)
    return df

dfs = [load_one(RAW/f) for f in FILES]

In [ ]:
# Intersect common columns across files to avoid mismatches
common = set(dfs[0].columns)
for df in dfs[1:]: common &= set(df.columns)
dfs = [df[list(common)] for df in dfs]
full = pd.concat(dfs, ignore_index=True)

label_col = 'Label' if 'Label' in full.columns else 'label'
y = full[label_col].astype(int).values
Xdf = full.drop(columns=[label_col]) 

In [5]:
# One-hot encode categorical columns
cat = [c for c in Xdf.columns if Xdf[c].dtype == 'object']
Xdf = pd.get_dummies(Xdf, columns=cat)

X = Xdf.values.astype(np.float32)
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

In [ ]:
np.save("BFIPPDSR_project/data/processed/cic_X_train.npy", X_train)
np.save("BFIPPDSR_project/data/processed/cic_y_train.npy", y_train)
np.save("BFIPPDSR_project/data/processed/cic_X_test.npy",  X_test)
np.save("BFIPPDSR_project/data/processed/cic_y_test.npy",  y_test)
print("CICIDS2017 processed and saved.")

CICIDS2017 processed and saved.
